In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import confusion_matrix, roc_auc_score


In [2]:
#uso os dados do pré-processamento

X_train_np = np.load("../artifacts/data/X_train.npy")
X_test_np  = np.load("../artifacts/data/X_test.npy")
y_train_np = np.load("../artifacts/data/y_train.npy")
y_test_np  = np.load("../artifacts/data/y_test.npy")

#confirmo os dados
print(X_train_np.shape, X_test_np.shape)


(175341, 194) (82332, 194)


In [3]:
#transformo a matriz NumPy em Tensores PyTorch
#permito que os cálculos sejam feitos a nível de GPU (+velocidade)
#float32: preciso o suficiente para o modelo aprender detalhes sutis dos ataques de rede e leve o suficiente
#para não elevar o consumo de memória do meu PC

#X: tensores de características (dados da rede). Alimentam a rede neural
#Y: são so tensores label (gabarito) que o PyTorch vai usar para calcular o erro e ajustar o modelo

X_train_t = torch.tensor(X_train_np, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_np, dtype=torch.float32)

y_train_t = torch.tensor(y_train_np, dtype=torch.float32)
y_test_t  = torch.tensor(y_test_np, dtype=torch.float32)


In [4]:
#mecanismo de entrega de dados para a rede neural

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t), #empacota os dados (pergunta e resposta)
    batch_size=256, #modelo olha 256 exemplos por vez
    shuffle=True #embaralha os dados a cada época de treinamento
)

#DataLoader foi escolhido por ser otimizado, pois gerencia a memória do PC automaticamente e permite processamento paralelo
#evitando gargalo de tempo

In [5]:
#uso de rede neural profunda

class BaselineMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), #recebe os dados da rede (194 características) e espalha para 128 neurônios. Aqui há busca de combinações simples entre os dados
            nn.ReLU(), #filtro de relevância desliga o neurônio caso envie um valor negativo
            nn.Linear(128, 64), #condensa o aprendizado de 128 neurônios em 64
            nn.ReLU(),
            nn.Linear(64, 1) #camada de saída. Reduz os 64 em um único neurônio
        )

    def forward(self, x):
        return self.net(x).squeeze() #ajuste de formato, entrega apenas número puro


In [6]:
#config de ambiente de hardware e motores de otimização

device = "cuda" if torch.cuda.is_available() else "cpu" #uso da GPU como prioridade

#carrego uma instância da rede neural para a memória da GPU (ou RAM) e garanto que a primeira camada da rede tenha o tamanho exato para receber
#os dados pré-processados
model = BaselineMLP(X_train_np.shape[1]).to(device)

criterion = nn.BCEWithLogitsLoss() #mede o quão longe a previsão do modelo está da realidade, ideal para um IDS
optimizer = optim.Adam(model.parameters(), lr=0.001) #otimizador Adam, pois possui taxa de aprendizado adaptativa. Rápido para aprender e lento para não ignorar detalhes importantes


In [7]:
#loop de treinamento

epochs = 10

for epoch in range(epochs):
    model.train() #treinamento
    total_loss = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad() #apaga o cálculo anterior a cada novo problema para não se confundir

        logits = model(xb) #palpite
        loss = criterion(logits, yb) #modelo compara o palpite com a resposta real. Maior a Loss, pior ele foi

        loss.backward() #modelo analisa o erro e tenta descobrir quais neurônios o levaram a resposta errada
        optimizer.step() #ajuste 

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}") #observo se está aprendendo


Epoch 1/10 - Loss: 0.1580
Epoch 2/10 - Loss: 0.1214
Epoch 3/10 - Loss: 0.1186
Epoch 4/10 - Loss: 0.1168
Epoch 5/10 - Loss: 0.1151
Epoch 6/10 - Loss: 0.1140
Epoch 7/10 - Loss: 0.1130
Epoch 8/10 - Loss: 0.1123
Epoch 9/10 - Loss: 0.1114
Epoch 10/10 - Loss: 0.1110


In [8]:
#aqui quero testar se o modelo com dados que ele nunca viu para saber se ele aprendeu a detectar ou se apenas decorou o BD

model.eval() #coloco o modelo em modo avaliação, aqui quero performance máxima

with torch.no_grad(): #economia de energia
    logits = model(X_test_t.to(device)) #passo os dados para o modelo
    probs = torch.sigmoid(logits).cpu().numpy() #transformando em probabilidade e tranformando Tensor em Numpy


In [9]:
threshold = 0.5 #pode ser ajustado conforme a necessidade de operação do IDS

y_pred = (probs >= threshold).astype(int)

#tp: diz ser ataque e é mesmo
#tn: dizer ser normal e é mesmo
#fp: diz ser ataque mas é normal (alarme falso)
#fn: diz ser normal mas é ataque (falha)

tn, fp, fn, tp = confusion_matrix(y_test_np, y_pred).ravel()

recall = tp / (tp + fn)
fpr    = fp / (fp + tn)
auc    = roc_auc_score(y_test_np, probs)

print(f"Recall : {recall:.2%}") #sensibildiade: porcentagem total que o modelo conseguiu pegar
print(f"FPR    : {fpr:.2%}") #alarme falso
print(f"ROC AUC: {auc:.3}") #de 0 a 1, as habilidades do modelo em separar as duas classes


Recall : 98.02%
FPR    : 29.55%
ROC AUC: 0.976
